# Описание полученного варианта

## Вариант 3 (№21 в списке): 
- Исследуемые поля:
    - timestamp
    - hour
    - is_weekend
    - device_type
    - os_family
    - browser_family
    - known_device
- Основная проверка:
Временное разделение по timestamp

In [1]:
# Подготовка проекта

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import sklearn

RANDOM_STATE = 42
DATA_PATH = Path('data/auth_events_lab01.xlsm')

print(sys.version)
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)

3.11.16 (main, Aug 13 2026, 09:46:36) [GCC 15.2.0]
pandas: 2.2.2
scikit-learn: 1.4.2


## Постановка задачи и паспорт датасета

### Прикладная задача ИБ

Требуется оценить, является ли конкретная попытка аутентификации подозрительной, используя контекст устройства и времени события (тип устройства, семейство ОС, семейство браузера, признак известного устройства, час суток, признак выходного дня). Результат предназначен для аналитика SOC / системы мониторинга ИБ: он позволяет приоритизировать проверки, повысить уровень логирования для подозрительных сессий, запросить дополнительный фактор аутентификации, ограничить доступ к чувствительным ресурсам или инициировать ручную верификацию события. Автоматическое блокирование учетной записи по одной метке не предполагается — решение носит характер поддержки принятия решений.

### Объект анализа и гранулярность наблюдения

**Объект анализа** — одна попытка аутентификации пользователя.

**Гранулярность наблюдения** — одна строка таблицы соответствует ровно одной попытке входа, зафиксированной средством защиты. 

Одна строка не соответствует одному пользователю или инциденту, так как:

- один пользователь за период наблюдения совершает много попыток, поэтому строка на пользователя потеряла бы событийный контекст;
- один инцидент может объединять несколько попыток (например, серию неуспешных входов с последующим успехом), поэтому строка на инцидент смешала бы разные моменты принятия решения;
- решение принимается **в момент попытки**, поэтому единица наблюдения — именно попытка.

### Целевая переменная `is_suspicious`

**Целевая переменная:** `is_suspicious` — бинарная метка, отражающая итоговую оценку события.

- `1` — событие признано **подозрительным** (повышенный риск);
- `0` — событие признано **нормальным** (риск в пределах допустимого).

Имя переменной `is_suspicious` не является аргументом в пользу корректности разметки. Оно задает только семантику целевого признака — заявленный смысл "подозрительное / нормальное событие", — но не подтверждает, что проставленные в наборе метки действительно соответствуют реальному классу объекта.

Корректность разметки должна обосновываться отдельно и независимо от названия столбца: через источник меток (кто или что их формирует) и через процедуру их получения (по какому правилу и в какой момент). Пока источник и процедура не проверены, имя `is_suspicious` остается лишь гипотезой о смысле признака, а не доказательством качества метки.

### Паспорт датасета

**Источник:** лист «Данные», из таблицы, прикрепленной к лабораторной работе (https://disk.yandex.ru/i/jROt0cPOXj_qHA).

**Дата получения:** 23.09.2026.

**Исходный размер:** 163 Kb

**Ограничения:** 

Набор синтетически сгенерированный, поэтому существует ряд ограничений:

- распределения признаков и метки могут не соответствовать реальному SOC;
- источник не содержит реальных персональных, конфиденциальных или опасных данных;
- правила генерации метки и признаков неизвестны, поэтому возможно наличие скрытых зависимостей между полями;
- часть полей (`analyst_verdict`, `analyst_risk_score`) появляется **после** события и не должна использоваться при принятии решения.

### Таблица признаков

| Столбец | Смысл | Фактический тип | Ожидаемый тип | Допустимые значения | Момент доступности | Решение |
|---|---|---|---|---|---|---|
| `event_id` | Технический идентификатор записи | string | string | уникальная строка | до события (служебный) | исключить |
| `timestamp` | Время попытки аутентификации | datetime | datetime | в пределах периода сбора | в момент события | использовать только для разделения |
| `user_id` | Псевдоним учетной записи | category | category | `usr_XXX` | в момент события | использовать как группу при анализе, из X исключить |
| `department` | Подразделение пользователя | category | category | IT, HR, SOC, R&D, Sales, Finance, Operations | в момент события | оставить как контекст |
| `source_ip` | IP-адрес источника | string | string | IPv4 | в момент события | оставить как контекст |
| `country` | Страна по IP | category | category | ISO-код | в момент события | оставить как контекст |
| `device_type` | Тип устройства | category | category | desktop, laptop, tablet, mobile | в момент события | оставить |
| `os_family` | Семейство ОС | category | category | Windows, macOS, Linux, iOS, Android | в момент события | оставить |
| `browser_family` | Семейство браузера | category | category | Chrome, Firefox, Safari, Edge | в момент события | оставить |
| `auth_method` | Способ аутентификации | category | category | password, otp, push, fido2 | в момент события | оставить как контекст |
| `hour` | Час суток по UTC | integer | integer | 0...23 | в момент события | оставить |
| `is_weekend` | Признак выходного дня | binary | binary | {0, 1} | в момент события | оставить |
| `failed_attempts_24h` | Неуспешных попыток за 24 ч | integer | integer | >= 0 | в момент события | оставить как контекст |
| `login_velocity_1h` | Попыток входа за последний час | integer | integer | >= 0 | в момент события | оставить как контекст |
| `distance_from_usual_km` | Расстояние от обычной геолокации | float | float | 0...20000 | в момент события | оставить как контекст |
| `account_age_days` | Возраст учетной записи, дни | integer | integer | >= 0 | в момент события | оставить как контекст |
| `password_age_days` | Возраст текущего пароля, дни | integer | integer | >= 0 | в момент события | оставить как контекст |
| `known_device` | Известно ли устройство системе | binary | binary | {0, 1} | в момент события | оставить |
| `new_country` | Новая ли страна для пользователя | binary | binary | {0, 1} | в момент события | оставить как контекст |
| `vpn_used` | Обнаружено ли использование VPN | binary | binary | {0, 1} | в момент события | оставить как контекст |
| `analyst_verdict` | Вердикт аналитика | category | category | benign, confirmed_threat | после решения | исключить (утечка) |
| `analyst_risk_score` | Оценка риска аналитиком | integer | integer | 0...100 | после решения | исключить (утечка) |
| `is_suspicious` | Целевая метка | binary | binary | {0, 1} | после события | это `y`, в `X` не входит |

- **`y`** — целевая переменная `is_suspicious`.
- **`X`** — матрица признаков: все столбцы, кроме `y` и полей, которые исключены по причине утечки.

In [ ]:
# Загрузка и первичная инвентаризация

df = pd.read_excel(DATA_PATH, sheet_name='Данные')

original_shape = df.shape
original_rows, original_cols = original_shape

print(f'Загружен файл: {DATA_PATH}')

print('=== Первые 5 строк ===')
display(df.head(5))

print('\n=== Последние 5 строк ===')
display(df.tail(5))

original_rows, original_cols = df.shape
print('=== Исходный размер таблицы ===')
print(f'Число строк:   {original_rows}')
print(f'Число столбцов: {original_cols}')

print('\n=== Названия столбцов ===')
for i, col in enumerate(df.columns, 1):
    print(f'{i:>2}. {col}')

print('\n=== Типы данных ===')
print(df.dtypes)

print('\n=== Объем памяти ===')
mem = df.memory_usage(deep=True)
print(f'\nИтого: {mem.sum() / 1024:.3f} Кб')

# Целевой столбец присутствует ровно один раз
target_count = (df.columns == 'is_suspicious').sum()
print(f'Столбцов с именем is_suspicious: {target_count}')
assert target_count == 1, 'Ожидался один столбец is_suspicious'